# Example 15.3 / 2: Check citations and code

Companion notebook for **15.3 AMPL in NotebookLM** in the AMPL lab Documentation, section "Check citations and code". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Check citations and code

> **A citation can still be misapplied**
> The cited passage may be genuine while the assistant applies it to a different API object, solver, version, or model class. Read enough surrounding context to confirm the connection.

A misapplied citation is easy to reproduce. The `set_data` passage is genuine, but it applies
only when every column names a parameter in the model. Here the DataFrame has a `price` column
while the model calls the parameter `profit`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;               # steel products
param rate {PROD} > 0;  # tons per hour
param profit {PROD};    # dollars per ton; the DataFrame has no profit column

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# This step fails on purpose, as on the page; the error is printed so the notebook goes on
try:
    # data.py
    import pandas as pd

    products = pd.DataFrame(
        {"rate": [200, 140], "price": [25, 30]},
        index=pd.Index(["bands", "coils"], name="PROD"),
    )
    ampl.set_data(products, "PROD")
except Exception as error:
    print(f"{type(error).__name__}: {error}")

RuntimeError: price is not a subscripted param
context:  data; param: PROD: rate price   >>> := <<<  'bands' 200 25 'coils' 140 30 


The rest of this run is left out: on the page it stops at the error above.